# DiffSinger Miadu Fine-tuning (路徑自動診斷版)
**新增功能**：
- 自動偵測並回報 Google Drive 中的檔案結構
- 強化對「空格資料夾名」的識別

## 第一步：掛載 Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 第二步：自動診斷 Drive 路徑 (解決 FileNotFoundError)

In [ ]:
import os
import glob

print("| 正在掃描 Google Drive...")
possible_roots = glob.glob("/content/drive/MyDrive/DiffSinger*Colab*")
if not possible_roots:
    print("❌ 錯誤：找不到名稱包含 'DiffSinger' 和 'Colab' 的資料夾！")
    print("請確認您在 Drive 根目錄創建了資料夾，且名稱正確。")
    drive_root = None
else:
    # 優先選擇最像的一個
    drive_root = possible_roots[0]
    print(f"✅ 找到路徑: '{drive_root}'")
    
    # 檢查子目錄
    print("| 資料夾內容:")
    for item in os.listdir(drive_root):
        print(f"  - {item}")

if drive_root:
    # 執行同步
    %cd /content/
    if not os.path.exists("diffsinger-repo"):
        !git clone https://github.com/shihte/DiffSinger-Miadu-Colab.git diffsinger-repo
    %cd diffsinger-repo
    
    !pip install --upgrade setuptools pip wheel
    !pip install praat-parselmouth pyloudnorm pypinyin pycwt pyworld lightning-flash==0.7.0
    
    !mkdir -p checkpoints data/binary
    !rm -rf checkpoints
    !ln -s "{drive_root}/checkpoints_finetune" checkpoints
    
    # 同步關鍵數據
    src_binary = os.path.join(drive_root, "miadu_finetune")
    if os.path.exists(src_binary):
        print(f"| 正在同步二進位數據...")
        !cp -r "{src_binary}" data/binary/
    else:
        print(f"❌ 錯誤：在 Drive 中找不到 '{src_binary}' 資料夾！")
        print("請確認您是否上傳了 'miadu_finetune' 資料夾到 Drive。")

## 第三步：啟動訓練

In [ ]:
import os
os.environ['PYTHONPATH'] = "."
!python tasks/run.py --config usr/configs/midi/e2e/miadu_finetune.yaml --exp_name miadu_finetune_v1